# PyTorch ile Derin Öğrenme · Eğitim döngüsü ve doğrulama

Macademia · Büyükler için

> **Colab'da çalıştırma:** Bu dosyayı [colab.research.google.com](https://colab.research.google.com) adresinde **Dosya → Not defteri yükle** ile aç. GPU için **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç. Hücreleri yukarıdan aşağı sırayla çalıştır (Shift + Enter). PyTorch Colab'da kuruludur; bilgisayarında çalıştırmak istersen `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # tekrarlanabilirlik
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Veri ve ayrım

In [ ]:
from torch import nn
from sklearn.datasets import make_moons
def moons(n, seed):                                                  # eğitim ve doğrulama ayrı üretilir
    X, y = make_moons(n, noise=0.2, random_state=seed)
    return torch.tensor(X, dtype=torch.float32).to(device), torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(device)
Xtr, ytr = moons(300, 0)
Xva, yva = moons(100, 1)
print(Xtr.shape, Xva.shape)

## 2. Dört adımlı döngü

In [ ]:
def make_model(h=16):
    return nn.Sequential(nn.Linear(2, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1)).to(device)

def train(model, epochs=100, lr=0.01, bs=32):
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.BCEWithLogitsLoss(); hist = []
    for epoch in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr)); tot = 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()                                          # 1) sıfırla
            loss = loss_fn(model(Xtr[idx]), ytr[idx])                # 2) ileri + kayıp
            loss.backward()                                          # 3) geri
            opt.step()                                               # 4) güncelle
            tot += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            va = loss_fn(model(Xva), yva).item(); acc = ((model(Xva) > 0).float() == yva).float().mean().item()
        hist.append((tot / len(Xtr), va, acc))
    return hist

hist = train(make_model())
print('son: train %.3f val %.3f acc %.3f' % hist[-1])

## 3. Eğriler ve karar sınırı

In [ ]:
import matplotlib.pyplot as plt
h = np.array(hist)
plt.plot(h[:, 0], label='train'); plt.plot(h[:, 1], label='val'); plt.legend(); plt.xlabel('epoch'); plt.show()

## 4. Duman testi: tek yığını ezberle

In [ ]:
m = make_model(64); opt = torch.optim.Adam(m.parameters(), lr=0.01)
xb, yb = Xtr[:32], ytr[:32]
for _ in range(300):
    opt.zero_grad(); l = nn.BCEWithLogitsLoss()(m(xb), yb); l.backward(); opt.step()
print('tek yığın kaybı:', l.item())                     # ≈ 0 olmalı

## Alıştırmalar
1. Öğrenme oranını 0,001 ve 0,1 yap; eğrileri karşılaştır.
2. Aynı tohumla iki kez çalıştır: sonuç aynı mı?
3. En iyi doğrulama kaybındaki modeli kaydeden bir checkpoint ekle.